# ⚙️ Quantize & Export to GGUF — Llama-3.1-70B Uncensored

**Goal:** Merge LoRA adapters → full model → quantize to GGUF Q4_K_M  
for optimal performance on **RTX 5090 (24 GB VRAM) + 64 GB RAM**.

## Strategy for RTX 5090 (24 GB) + 64 GB RAM + Intel i9 275X
| Quant | File size | GPU layers | Speed (tok/s) | Quality |
|---|---|---|---|---|
| Q5_K_M | ~51 GB | ~22/80 | ~12–15 | Excellent |
| **Q4_K_M** | **~42 GB** | **~28/80** | **~18–22** | **Very good ✅** |
| Q3_K_M | ~35 GB | ~35/80 | ~25–30 | Good |
| IQ3_M  | ~32 GB | ~38/80 | ~28–35 | Good (imatrix) |

**Recommendation:** Q4_K_M — best quality/speed trade-off.  
Load with `llama.cpp` using `-ngl 28` (28 transformer layers on GPU, rest on CPU RAM).

In [ ]:
# ── Step 0: GPU check & install ─────────────────────────────────────────────
!nvidia-smi
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q huggingface_hub transformers sentencepiece

In [ ]:
# ── Step 1: Authenticate & mount Drive ─────────────────────────────────────
import os
from getpass import getpass
from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")

HF_TOKEN  = getpass("🔑 HuggingFace token (needs write access for push): ")
HF_REPO_ID = input("🔑 HuggingFace repo to push GGUF to (e.g. username/llama31-70b-uncensored-GGUF): ").strip()
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

BASE_MODEL   = "meta-llama/Llama-3.1-70B-Instruct"
LORA_PATH    = "/content/drive/MyDrive/AI-command-center/checkpoints/llama31-70b-uncensored/lora_adapters_final"
MERGED_DIR   = "/content/drive/MyDrive/AI-command-center/merged_model"
GGUF_DIR     = "/content/drive/MyDrive/AI-command-center/gguf"
QUANT_TYPES  = ["q4_k_m", "q5_k_m"]  # export both; pick one for local use

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(GGUF_DIR,   exist_ok=True)
print("✅ Paths ready")

In [ ]:
# ── Step 2: Load base model + LoRA adapters ──────────────────────────────────
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = LORA_PATH,
    max_seq_length = 4096,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
print(f"✅ Loaded fine-tuned model from {LORA_PATH}")

In [ ]:
# ── Step 3: Merge LoRA adapters into full fp16 weights & save ────────────────
# This creates a standard HuggingFace model you can load with transformers.
# The merged model is ~140 GB on disk (fp16) — make sure you have Drive space.
print("Merging LoRA adapters into base model weights (fp16)...")
print("⚠️  This will use ~80 GB RAM. If you get OOM errors, use the fp32→fp16 trick below.")

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method = "merged_16bit",  # full fp16 merge
)
print(f"✅ Merged model saved to {MERGED_DIR}")

In [ ]:
# ── Step 4: Export directly to GGUF via Unsloth ──────────────────────────────
# Unsloth can export directly without going through the merged fp16 step
# which saves time and disk space. Use this if Step 3 caused OOM.

for quant in QUANT_TYPES:
    out_file = f"{GGUF_DIR}/llama31-70b-uncensored-{quant.upper()}.gguf"
    print(f"\n⏳ Exporting {quant.upper()} GGUF → {out_file}")
    model.save_pretrained_gguf(
        GGUF_DIR + f"/llama31-70b-uncensored-{quant}",
        tokenizer,
        quantization_method = quant,
    )
    print(f"✅ Done: {quant.upper()}")

import os
print("\nGGUF files:")
for f in os.listdir(GGUF_DIR):
    size_gb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1e9
    print(f"  {f:60s} {size_gb:.1f} GB")

In [ ]:
# ── Step 5: Push GGUF files to HuggingFace Hub ───────────────────────────────
# HuggingFace supports large files via LFS automatically.
from huggingface_hub import HfApi
import os

api = HfApi(token=HF_TOKEN)

# Create the repo if it doesn't exist
api.create_repo(
    repo_id    = HF_REPO_ID,
    repo_type  = "model",
    exist_ok   = True,
    private    = True,   # keep private until you're satisfied
)

# Upload all GGUF files
for fname in os.listdir(GGUF_DIR):
    if fname.endswith(".gguf"):
        fpath = os.path.join(GGUF_DIR, fname)
        print(f"Uploading {fname} ({os.path.getsize(fpath)/1e9:.1f} GB)...")
        api.upload_file(
            path_or_fileobj = fpath,
            path_in_repo    = fname,
            repo_id         = HF_REPO_ID,
            repo_type       = "model",
        )
        print(f"  ✅ Uploaded {fname}")

print(f"\n✅ All files at: https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# ── Step 6: Create a model card README ──────────────────────────────────────
model_card = f"""---
base_model: {BASE_MODEL}
library_name: gguf
tags:
- llama
- qlora
- uncensored
- gguf
---

# Llama-3.1-70B Uncensored (GGUF)

QLoRA fine-tune of [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL})  
on [ZombitX64/UncensoredOssbit](https://huggingface.co/datasets/ZombitX64/UncensoredOssbit).

## Recommended local usage (RTX 5090 24 GB + 64 GB RAM)

```bash
# Download Q4_K_M (best quality/speed trade-off)
ollama pull {HF_REPO_ID}:Q4_K_M
# or with llama.cpp:
./llama-cli -m llama31-70b-uncensored-Q4_K_M.gguf \\
  --n-gpu-layers 28 -c 4096 -t 24 --color
```

## Files
| File | Quant | Size | GPU layers (24 GB VRAM) |
|---|---|---|---|
| Q4_K_M.gguf | Q4_K_M | ~42 GB | 28 / 80 |
| Q5_K_M.gguf | Q5_K_M | ~51 GB | 22 / 80 |
"""

card_path = "/tmp/README.md"
with open(card_path, "w") as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj = card_path,
    path_in_repo    = "README.md",
    repo_id         = HF_REPO_ID,
    repo_type       = "model",
)
print("✅ Model card uploaded")

## ✅ Done!

Your quantized GGUF model is ready.  
**Next steps:**  
1. Download the GGUF file to your Windows PC.  
2. Follow `INSTALLATION.md` in the repo to set up llama.cpp, Ollama, and the AI Command Center.

**Recommended download command on Windows (PowerShell):**
```powershell
huggingface-cli download YOUR-HF-USERNAME/llama31-70b-uncensored-GGUF `
    llama31-70b-uncensored-Q4_K_M.gguf `
    --local-dir C:\AI-models\
```